In [7]:
import torch
import os
import sys
import argparse
import numpy as np

helpers_path = os.path.join('/ether/aegis/Research_HEP/NRAD/oldver/NRAD/non-resonant-AD/model_scripts')
sys.path.insert(0, os.path.abspath(helpers_path))
from Classifier import Classifier
from SimpleMAF import SimpleMAF

In [8]:
seed = 2
data_path = f"SemiVisJets/data/data_seed{seed}"
mc_path = "SemiVisJets/data"
model_path = "SemiVisJets/models"
# config_path = "oldver/NRAD/non-resonant-AD/Train_Models/configs"
samples_path = "SemiVisJets/samples"

In [9]:
print("Setting up device...")
CUDA = torch.cuda.is_available()
print("cuda available:", CUDA)
device = torch.device("cuda" if CUDA else "cpu")

Setting up device...
cuda available: True


In [10]:
print("Test Datasets")
n_context = 2
mc_events = np.load(f"{mc_path}/mc_events_chunk{1:02d}.npz", allow_pickle=True)
mc_events_cr = mc_events["mc_events_cr"]
mc_events_sr = mc_events["mc_events_sr"]
print(mc_events_cr.shape, mc_events_sr.shape)
data_chunk = np.load(f"SemiVisJets/data/data_test/data_events_chunk{6:02d}.npz", allow_pickle=True)
data_events_cr = data_chunk["data_events_cr"]
data_events_sr = data_chunk["data_events_sr"]
print(data_events_cr.shape, data_events_sr.shape)

Test Datasets
(9953032, 7) (46968, 7)
(9983635, 7) (16365, 7)


In [11]:

for i in range(1, 6):
    print("Loading data chunk", i)
    data_events_cr = data_chunk["data_events_cr"]
    print("CR has", len(data_events_cr), "data events,", len(mc_events_cr), "MC events.")


    data_cr_test = data_events_cr[:, :n_context]
    mc_cr_test = mc_events_cr[:, :n_context]
    mc_events_sr_test = mc_events_sr[:, :n_context]
    print("Loading model... at data chunk", i)
    model = "context_weight"
    model_path_full = f"{model_path}/{model}_MC{seed:02d}_Data{i:02d}.pt"
    print("Model path:", model_path_full)
    NN_Context_Weights = torch.load(model_path_full, weights_only=False)
    NN_Context_Weights.to(device)

    print("Generating samples... at data chunk", i)
    w_cr = NN_Context_Weights.evaluation(mc_cr_test)
    w_cr = (w_cr/(1-w_cr)).flatten()

    np.savez_compressed(f"{samples_path}/{model}_MC{seed:02d}_Data{i:02d}_CR_samples.npz", target_cr = data_cr_test, mc_cr = mc_cr_test, w_cr = np.nan_to_num(w_cr, copy=False, nan=0.0, posinf=0.0, neginf=0.0))
    print("Saved samples to", f"{samples_path}/{model}_MC{seed:02d}_Data{i:02d}_CR_samples.npz")
    w_sr = NN_Context_Weights.evaluation(mc_events_sr_test)
    w_sr = (w_sr/(1-w_sr)).flatten()
    np.savez_compressed(f"{samples_path}/{model}_MC{seed:02d}_Data{i:02d}_SR_samples.npz", mc_samples = mc_events_sr, w_sr = np.nan_to_num(w_sr, copy=False, nan=0.0, posinf=0.0, neginf=0.0))
    print("Saved samples to", f"{samples_path}/{model}_MC{seed:02d}_Data{i:02d}_SR_samples.npz")

print("Done!")


Loading data chunk 1
CR has 9983635 data events, 9953032 MC events.
Loading model... at data chunk 1
Model path: SemiVisJets/models/context_weight_MC02_Data01.pt
Generating samples... at data chunk 1
Saved samples to SemiVisJets/samples/context_weight_MC02_Data01_CR_samples.npz
Saved samples to SemiVisJets/samples/context_weight_MC02_Data01_SR_samples.npz
Loading data chunk 2
CR has 9983635 data events, 9953032 MC events.
Loading model... at data chunk 2
Model path: SemiVisJets/models/context_weight_MC02_Data02.pt
Generating samples... at data chunk 2
Saved samples to SemiVisJets/samples/context_weight_MC02_Data02_CR_samples.npz
Saved samples to SemiVisJets/samples/context_weight_MC02_Data02_SR_samples.npz
Loading data chunk 3
CR has 9983635 data events, 9953032 MC events.
Loading model... at data chunk 3
Model path: SemiVisJets/models/context_weight_MC02_Data03.pt
Generating samples... at data chunk 3
Saved samples to SemiVisJets/samples/context_weight_MC02_Data03_CR_samples.npz
Saved

In [ ]:
for i in range(1, 6):
    print("CR has", len(data_events_cr), "data events,", len(mc_events_cr), "MC events.")

    data_cr_test = data_events_cr
    data_sr_test = data_events_sr
    mc_cr_test = mc_events_cr

    print("Loading model... at data chunk", i)
    model = "reweight"
    model_path_full = f"{model_path}/{model}_MC{seed:02d}_Data{i:02d}.pt"
    print("Model path:", model_path_full)
    NN_Reweight = torch.load(model_path_full, weights_only=False)
    NN_Reweight.to(device)

    print("Generating samples... at data chunk", i)
    w_cr = NN_Reweight.evaluation(mc_cr_test)
    w_cr = (w_cr/(1-w_cr)).flatten()

    np.savez_compressed(f"{samples_path}/{model}_MC{seed:02d}_Data{i:02d}_CR_samples.npz", target_cr = data_cr_test, mc_cr = mc_cr_test, w_cr = np.nan_to_num(w_cr, copy=False, nan=0.0, posinf=0.0, neginf=0.0))
    print("Saved samples to", f"{samples_path}/{model}_MC{seed:02d}_Data{i:02d}_CR_samples.npz")

    mc_events_sr = mc_events_sr
    w_sr = NN_Reweight.evaluation(mc_events_sr)
    w_sr = (w_sr/(1-w_sr)).flatten()
    np.savez_compressed(f"{samples_path}/{model}_MC{seed:02d}_Data{i:02d}_SR_samples.npz", data_sr = data_sr_test, mc_samples = mc_events_sr, w_sr = np.nan_to_num(w_sr, copy=False, nan=0.0, posinf=0.0, neginf=0.0))
    print("Saved samples to", f"{samples_path}/{model}_MC{seed:02d}_Data{i:02d}_SR_samples.npz")

print("Done!")


CR has 9983635 data events, 9953032 MC events.
Loading model... at data chunk 1
Model path: SemiVisJets/models/reweight_MC02_Data01.pt
Generating samples... at data chunk 1
Saved samples to SemiVisJets/samples/reweight_MC02_Data01_CR_samples.npz
Saved samples to SemiVisJets/samples/reweight_MC02_Data01_SR_samples.npz
CR has 9983635 data events, 9953032 MC events.
Loading model... at data chunk 2
Model path: SemiVisJets/models/reweight_MC02_Data02.pt
Generating samples... at data chunk 2
Saved samples to SemiVisJets/samples/reweight_MC02_Data02_CR_samples.npz
Saved samples to SemiVisJets/samples/reweight_MC02_Data02_SR_samples.npz
CR has 9983635 data events, 9953032 MC events.
Loading model... at data chunk 3
Model path: SemiVisJets/models/reweight_MC02_Data03.pt
Generating samples... at data chunk 3
Saved samples to SemiVisJets/samples/reweight_MC02_Data03_CR_samples.npz
Saved samples to SemiVisJets/samples/reweight_MC02_Data03_SR_samples.npz
CR has 9983635 data events, 9953032 MC even

In [13]:
import torch, numpy as np, gc

def safe_sample(maf, cond, batch_size=1000, num_samples=1):
    preds = []
    for j in range(0, len(cond), batch_size):
        cond_batch = cond[j:j+batch_size]
        with torch.no_grad():
            preds.append(maf.sample(num_samples, cond_batch))
        # Free memory after each batch
        torch.cuda.empty_cache()
        gc.collect()
    return np.concatenate(preds, axis=0)


In [14]:
n_context = 2
device = torch.device("cuda")
for i in range(1, 6):
    print("Loading data chunk", i)
    print("CR has", len(data_events_cr), "data events,", len(mc_events_cr), "MC events.")

    data_context_cr_test = data_events_cr[:, :n_context]
    data_feature_cr_test = data_events_cr[:, n_context:]
    data_feature_sr_test = data_events_sr[:, n_context:]
    mc_context_sr = mc_events_sr[:, :n_context]

    print("Loading model... at data chunk", i)
    model = "generate"
    model_path_full = f"{model_path}/{model}_MC{seed:02d}_Data{i:02d}.pt"
    MAF = torch.load(model_path_full, weights_only=False)
    MAF.to(device)

    print("Generating samples... at data chunk", i)

    # 🔹 Batched sampling for CR
    pred_bkg_CR = safe_sample(MAF, data_context_cr_test, batch_size=500)
    np.savez(f"{samples_path}/{model}_MC{seed:02d}_Data{i:02d}_CR_samples.npz",
             target_cr=data_feature_cr_test, generate_cr=pred_bkg_CR)

    # 🔹 Batched sampling for SR
    oversample = 1
    pred_bkg_SR = safe_sample(MAF, mc_context_sr, batch_size=500, num_samples=oversample)
    np.savez(f"{samples_path}/{model}_MC{seed:02d}_Data{i:02d}_SR_samples.npz",
             data_sr=data_feature_sr_test, samples=pred_bkg_SR)

    print(f"Saved CR & SR samples for chunk {i}")
    torch.cuda.empty_cache()



Loading data chunk 1
CR has 9983635 data events, 9953032 MC events.
Loading model... at data chunk 1
Generating samples... at data chunk 1
Saved CR & SR samples for chunk 1
Loading data chunk 2
CR has 9983635 data events, 9953032 MC events.
Loading model... at data chunk 2
Generating samples... at data chunk 2
Saved CR & SR samples for chunk 2
Loading data chunk 3
CR has 9983635 data events, 9953032 MC events.
Loading model... at data chunk 3
Generating samples... at data chunk 3
Saved CR & SR samples for chunk 3
Loading data chunk 4
CR has 9983635 data events, 9953032 MC events.
Loading model... at data chunk 4
Generating samples... at data chunk 4
Saved CR & SR samples for chunk 4
Loading data chunk 5
CR has 9983635 data events, 9953032 MC events.
Loading model... at data chunk 5
Generating samples... at data chunk 5
Saved CR & SR samples for chunk 5
